In [83]:
import os 
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display,Markdown,update_display
from bs4 import BeautifulSoup
import json
import logging

In [84]:
load_dotenv()

True

In [85]:
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)


In [86]:
openai = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GEMINI_API_KEY")
)

In [87]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [88]:
from urllib.parse import urljoin

def fectch_websiite_links(url):
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    links = []

    for tag in soup.find_all("a"):
        href = tag.get("href")

        if href:
            href = urljoin(url, href)

            if not (
                href.startswith("mailto:")
                or href.startswith("tel:")
                or href.startswith("javascript:")
                or "#" in href
            ):
                links.append(href)

    return list(set(links))

In [89]:
def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg", "img"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)

    return text

In [90]:
link_system_prompt = """
You are an expert website navigator.

Your task is to analyze a list of URLs extracted from a company's website and identify ONLY the pages that contain important information for generating a professional company brochure.

Return ONLY a valid JSON object.

Rules:
- Only include links that are likely to contain useful company information.
- Always return absolute URLs.
- Do not invent or modify URLs.
- Do not include duplicate URLs.
- Ignore anchors (#), mailto links, tel links, javascript links, PDFs, images, videos, login pages, cart pages, search pages, privacy/legal pages, cookie pages, terms pages, or social media links unless they are the company's primary contact page.
- If multiple URLs point to the same section, keep the best one.
- If a page belongs to more than one category, choose the most relevant category.
- If no useful links exist, return:
{
  "links": []
}

Output format:

{
  "links": [
    {
      "type": "<category>",
      "url": "<absolute_url>"
    }
  ]
}

Example 1

Input:
[
  "https://acme.com/",
  "https://acme.com/about",
  "https://acme.com/products",
  "https://acme.com/blog",
  "https://acme.com/privacy",
  "https://twitter.com/acme"
]

Output:
{
  "links": [
    {
      "type": "homepage",
      "url": "https://acme.com/"
    },
    {
      "type": "about",
      "url": "https://acme.com/about"
    },
    {
      "type": "products",
      "url": "https://acme.com/products"
    },
    {
      "type": "blog",
      "url": "https://acme.com/blog"
    }
  ]
}

Return ONLY the JSON object.
"""

In [91]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company.
Respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
"""

    links = fectch_websiite_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [92]:
def select_relevent_links(url):

    response = openai.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {
                "role": "system",
                "content": link_system_prompt
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            }
        ],
        response_format={
            "type": "json_object"
        }
    )

    result = response.choices[0].message.content

    result = result.replace("```json", "")
    result = result.replace("```", "")
    result = result.strip()

    return json.loads(result)

In [93]:
def fetch_page_and_all_relevant_links(url):

    contents = fetch_website_contents(url)

    relevant_links = select_relevent_links(url)

    result = f"## Landing Page\n\n{contents}\n\n"

    result += "## Relevant Pages\n"

    for link in relevant_links["links"]:

        result += f"\n\n### {link['type']}\n"

        try:
            result += fetch_website_contents(link["url"])
        except Exception:
            pass

    return result

In [94]:
brochure_system_prompt = """
You are an expert business analyst specializing in transforming raw website content into polished, reader-friendly company brochures.

You will receive text extracted from one or more pages of a company's website.

Your goals are to:
- Understand what the company does.
- Identify its products, services, customers, markets, and differentiators.
- Capture its mission, values, and company culture when available.
- Summarize hiring opportunities and career information.
- Produce a brochure that is accurate, concise, and easy to read.

When analyzing the content:
- Merge duplicate information from different pages.
- Ignore navigation menus, headers, footers, cookie banners, legal pages, and other boilerplate.
- Prioritize factual business information over promotional language.
- Preserve important names, statistics, certifications, partnerships, and achievements.
- Do not invent or infer facts that are not explicitly stated.

Write for an audience that may include:
- Potential customers
- Investors
- Job candidates
- Business partners

Structure the brochure with clear Markdown headings. Include relevant sections such as:
- Company Overview
- Products & Services
- Industries Served
- Value Proposition
- Customers
- Company Culture
- Careers
- Contact Information

Omit any section for which reliable information is unavailable.

Style guidelines:
- Professional and engaging.
- Concise but informative.
- Avoid marketing hype.
- Avoid repetition.
- Use bullet points where they improve readability.
- Keep the total length between 400 and 700 words.

Output only the brochure in Markdown.
"""

In [95]:
def get_brochure_user_prompt(company_name, url):

    user_prompt = f"""
You are looking at a company called {company_name}.

Here are the contents of its landing page and other relevant pages.

Create a professional brochure in Markdown.

"""

    user_prompt += fetch_page_and_all_relevant_links(url)

    return user_prompt[:5000]

In [96]:
def create_brochure(company_name, url):

    response = openai.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_user_prompt(company_name, url)
            }
        ]
    )

    display(Markdown(response.choices[0].message.content))

In [97]:
def stream_brochure(company_name, url):

    stream = openai.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_user_prompt(company_name, url)
            }
        ],
        stream=True
    )

    response = ""

    handle = display(Markdown(""), display_id=True)

    for chunk in stream:

        delta = chunk.choices[0].delta.content

        if delta is not None:
            response += delta

            update_display(
                Markdown(response),
                display_id=handle.display_id
            )

In [98]:
stream_brochure(
    "HuggingFace",
    "https://huggingface.co"
)

INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


# Company Overview

Hugging Face is the leading global platform for the machine learning community, serving as the central hub where researchers, developers, and organizations collaborate to build the future of artificial intelligence. By providing an open-source ecosystem, Hugging Face enables the seamless creation, discovery, and deployment of machine learning models, datasets, and applications. 

With a vast repository containing over 2 million models and 500,000 datasets, the platform supports a wide range of modalities, including text, image, video, audio, and 3D. Hugging Face is the collaborative home for more than 50,000 organizations, ranging from non-profits and research institutions to the world's largest technology companies.

# Products & Services

Hugging Face provides a comprehensive stack of tools designed to accelerate the machine learning lifecycle:

*   **Core Platform (The Hub):** A collaborative environment to host, share, and discover public and private models, datasets, and applications (Spaces).
*   **Open-Source Libraries:** Hugging Face maintains the foundation of modern ML tooling, including:
    *   *Transformers & Diffusers:* State-of-the-art models for PyTorch.
    *   *PEFT & TRL:* Tools for parameter-efficient fine-tuning and reinforcement learning.
    *   *Tokenizers & Safetensors:* Optimized, secure utilities for high-performance ML.
    *   *Transformers.js:* Capability to run state-of-the-art ML directly in a web browser.
*   **Enterprise Solutions:** Tailored for business-critical needs, offering:
    *   **Enterprise Hub:** Includes enterprise-grade security, access controls, audit logs, and SSO.
    *   **Inference Endpoints:** Optimized deployment for models, starting at $0.60/hour for GPU compute.
    *   **Inference Providers:** Access to over 45,000 models via a unified API with no service fees.
    *   **Hugging Face PRO:** Personalized support and advanced features for power users.

# Industries Served

Hugging Face’s platform is industry-agnostic, providing the infrastructure required by any sector looking to integrate AI into its operations. Their community and enterprise user base includes:
*   **Technology Giants:** Meta, Google, Microsoft, Amazon, and Intel.
*   **Research & Non-Profits:** Organizations like Ai2 (Allen Institute for AI).
*   **SaaS & Enterprise Software:** Companies such as Grammarly and Writer.

# Value Proposition

Hugging Face distinguishes itself through its commitment to an "open" AI philosophy. By centralizing machine learning resources, they enable:
*   **Rapid Innovation:** Developers can leverage state-of-the-art pre-trained models rather than building from scratch.
*   **Seamless Collaboration:** Teams can share work, build portfolios, and engage with a global community.
*   **Scalability:** From hobbyist projects to enterprise-grade production environments, Hugging Face provides the infrastructure to scale seamlessly.

# Company Culture

Hugging Face is defined by its community-first approach. Their culture is centered on the democratization of AI, fostering a transparent environment where knowledge is shared freely. By building the "foundation of ML tooling" alongside the open-source community, the company emphasizes collaborative progress over proprietary silos.

# Careers

Hugging Face is committed to building the future of AI and regularly seeks talented individuals to join their mission. They offer a collaborative and innovative work environment for engineers, researchers, and business professionals. Interested candidates are encouraged to view current opportunities on their official careers page.

# Contact Information

To learn more about the platform, explore their documentation, or contact their enterprise support teams, visit the official website at [huggingface.co](https://huggingface.co). Engage with the community via their Discord server, GitHub repository, or the official company forum.